# AY128 Lecture 09: Reports and Packaging

### Feb. 19, 2026

# Agenda

- Introduction (this notebook, *00_Introduction*)
- Examples of other reports: ideas for plots, tables, structure, etc.
- Walk through of work-flow for packaging, testing, revision controlling, and distributing code

<center>
<h1>Python Application Building</h1>
<img src="http://i.imgur.com/91PUPZA.png" width=20%>
</center>

<center>
AY 128, Spring 2026 — Josh Bloom
</center>

[PyPI](https://pypi.org/) is the main package repository for Python

In [ ]:
!uv pip install requests --target=/tmp/

In [ ]:
!ls /tmp/requests*

`...dist-info` directory is part of the "standard infrastructure to manage project distributions installed on a system, so all tools that are installing or removing projects are interoperable." [PEP-376](https://www.python.org/dev/peps/pep-0376/#id19)

[Note the .whl extension...see ["Wheel vs EGG"](https://packaging.python.org/en/latest/discussions/wheel-vs-egg/) and [Wheel info](https://wheel.readthedocs.io/en/latest/)]

Now, you can have Python know about your special installation directory by modifying your `PYTHONPATH` environment variable in your `.bashrc`, `.cshrc`, or `.tcshrc` file:
```bash
#BASH Style: 
export PYTHONPATH=/tmp/requests:$PYTHONPATH
#CSH Style:
setenv PYTHONPATH /path/to/my_choice:$PYTHONPATH
```

# Getting and Installing Packages with `uv`

[`uv`](https://docs.astral.sh/uv/) is a modern, extremely fast Python package and project manager written in Rust. It replaces `pip`, `venv`, `pip-tools`, and more in a single tool.

### Installing packages

```bash
# install a package into the current environment
$ uv pip install requests

# install from a pyproject.toml
$ uv pip install -r pyproject.toml
```

### Working with projects

When you have a `pyproject.toml`, `uv` manages everything for you:

```bash
# add a dependency to your project
$ uv add requests

# sync your environment to match pyproject.toml + uv.lock
$ uv sync

# run a command inside the project environment
$ uv run python myscript.py
```

### Editable installs (for development)

```bash
$ uv pip install -e .
```

> **Historical note:** The traditional tool for installing packages is `pip` (`pip install requests`). You'll still see `pip` in many tutorials and older codebases. `uv` is a drop-in replacement that is much faster and also handles environments and lockfiles.

# Managing Environments with `uv`

* Open Source software is constantly changing - how do you protect working code against future updates?
* Or, what if there is a beta release of a package you want to try, but you don't want to fully commit yet?
* Virtual environments create a local, self-contained, and totally separate Python installation.
* Use them to create a local Python ecosystem, separate from your computer's main system, so that you can do what you want in one without affecting the other.

## `uv venv` — create virtual environments

```bash
# Create a new environment (defaults to .venv/)
$ uv venv

# Create with a specific Python version
$ uv venv --python 3.13

# Create with a custom name
$ uv venv my_project_env
```

> **Note:** `python -m venv myenv` is the stdlib way to create virtual environments. `uv venv` is faster and also lets you specify Python versions easily.

## Activating an environment and installing packages

```bash
# Activate the environment
$ source .venv/bin/activate
(.venv)$ which python
.venv/bin/python

# Install packages into the environment
(.venv)$ uv pip install numpy pandas requests

# Deactivate when done
(.venv)$ deactivate
```

Just delete to remove the environment:
```bash
$ rm -rf .venv
```

## Lockfiles — reproducible environments

Instead of `pip freeze > requirements.txt`, modern Python uses **lockfiles**.

When you use `uv` with a `pyproject.toml`, it automatically generates a `uv.lock` file that pins every dependency (including transitive ones) to exact versions:

```bash
# Initialize a project (creates pyproject.toml)
$ uv init my_project
$ cd my_project

# Add dependencies
$ uv add numpy pandas

# uv.lock is automatically created/updated
# To recreate the environment from the lockfile:
$ uv sync
```

The `uv.lock` file should be committed to version control — anyone can then run `uv sync` to get an identical environment.

## Historical note: `conda` and `poetry`

**conda** (via Anaconda/Miniconda) was long the standard in scientific Python. It manages both Python packages and non-Python dependencies (C libraries, etc.) via its own package repository. You may still encounter `conda` environments in older projects. See https://www.anaconda.com/

**poetry** was a popular modern package manager that introduced lockfiles and `pyproject.toml`-based workflows before `uv`. See https://python-poetry.org/

**`uv`** has largely superseded both for pure-Python projects: it's faster, handles environments + dependencies + building + publishing in one tool, and is compatible with the standard `pyproject.toml` format.

<center><h1> Command Line Parsing</h1></center>

```bash
python myawesomeprogram.py -o option1 -p parameter2 -Q -R
```

**Goal**: build a command-line 'standalone' codebase in Python, w/ CL options & keywords
 
 **Solution**: `argparse` (https://docs.python.org/3/library/argparse.html)
 
* Allows for  user-friendly command line interfaces, and leaves it up to the code to determine what it was the user wanted.

* Also automatically generates help & usage messages and issues errors when invalid arguments are provided.

(Note on `optparse`: being replaced in favor of `argparse`)

In [ ]:
import argparse

# Setting up a parser #


* First step for `argparse`: create parser object & tell it what arguments to expect. 
* It can then be used to process the command line arguments on runtime
* Parser class: `ArgumentParser`. Takes several arguments to set up the description used in the help text for the program & other global behaviors 
   
 <p>
See  http://www.doughellmann.com/PyMOTW/argparse/
</p>

In [ ]:
%%writefile myfile.py
#!/usr/bin/env python
import argparse
parser = argparse.ArgumentParser(description='Sample Application')
print("hi")

In [ ]:
%run myfile.py

In [ ]:
!chmod 700 myfile.py

In [ ]:
!./myfile.py

# Defining Arguments & Parsing

* Arguments can trigger different actions, specified by the action argument to `add_argument()`. 
* Several supported actions.
* Once all of the arguments are defined, you can parse the command line by passing a sequence of argument strings to `parse_args()`. 
* By default, arguments are taken from `sys.argv[1:]`, but you can also pass your own list.

In [ ]:
%%file argparse_action.py
import argparse
parser = argparse.ArgumentParser(description='Sample Application')
parser.add_argument('required_arg_1', help='This positional argument is required')
parser.add_argument('required_arg_2', help='This positional argument is also required')
parser.add_argument('-s', action='store', dest='simple_value',
                    help='Store a simple value')
parser.add_argument('-c', action='store_const', dest='constant_value',
                    const='value-to-store',
                    help='Store a constant value')
parser.add_argument('-t', action='store_true', default=False,
                    dest='boolean_switch',
                    help='Set a switch to true')
parser.add_argument('-a', action='append', dest='collection',
                    default=[],
                    help='Add repeated values to a list',
                    )
parser.add_argument('-A', action='append_const', dest='const_collection',
                    const='value-1-to-append',
                    default=[],
                    help='Add different values to list')
parser.add_argument('-B', action='append_const', dest='const_collection',
                    const='value-2-to-append',
                    help='Add different values to list')
parser.add_argument('--version', action='version', version='%(prog)s 1.0')

results = parser.parse_args()
print('required_args    =', results.required_arg_1, results.required_arg_2)
print('simple_value     =', results.simple_value)
print('constant_value   =', results.constant_value)
print('boolean_switch   =', results.boolean_switch)
print('collection       =', results.collection)
print('const_collection =', results.const_collection)

In [ ]:
%run argparse_action.py --help

* `store`: Save the value, after optionally converting it to a different type (default)
* `store_const`: Save the value as defined as part of the argument specification, rather than a value that comes from the arguments being parsed
* `store_true`/`store_false`: Save the appropriate boolean value
* `append`: Save the value to a list.  Multiple values are saved if the argument is repeated
* `append_const`: Save a value defined in the argument specification to a list
* `version`: Prints version details about the program and then exits

#!uv pip install click fire

In [ ]:
#!pip install click fire

In [ ]:
%%writefile hello-click.py
import click
@click.command()
@click.option('--count', default=1, help='number of greetings')
@click.argument('name')
def hello(count, name):
    for x in range(count):
        click.echo('Hello, %s!' % name)

if __name__ == '__main__':
    hello()

In [ ]:
!python hello-click.py --help

In [ ]:
!python hello-click.py --count=4 Cal

In [ ]:
%%writefile hello-fire.py
import fire

def hello(count, name):
    "number of greetings"
    for x in range(count):
        print('Hello, %s!' % name)

if __name__ == '__main__':
    fire.Fire(hello)

In [ ]:
!python hello-fire.py --help

In [ ]:
!python hello-fire.py --count=3 "AY128 Class"

# Breakout! #

* Go to the breakout folder in: `breakouts/`

* Work on the file `breakout1.py`.  Do not move or modify the other files, in the other folders, but you will need to use them.  (You may add files to these directories, if necessary)

* Build up a command line parser which allows the user to specify:
 - how many datapoints to generate
 - whether to plot with a filled in histogram or an outlined one
 - the title of the plot
 - And then have the plot be generated.

* We want to be able to run a command like:

```bash
python breakout1.py -t -n 200 -T "My Awesome Title"
```

In [ ]:
cd breakout/sol

In [ ]:
!cat plotting/histOutline.py

In [ ]:
%matplotlib inline

In [ ]:
%run breakout1_solution.py -t

In [ ]:
%run breakout1_solution.py -n 200 -T "My Awesome Title"

In [ ]:
%run breakout1_solution.py --help

In [ ]:
%load_ext watermark

In [ ]:
%watermark --iversions